<a href="https://colab.research.google.com/github/nursnaaz/zero-to-genai-engineer/blob/main/00_search_engine/notebooks/02_tfidf_explained.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TF-IDF — From Zero to Search Engine

**What you'll learn in this notebook:**
- Why simple word counting fails for search
- What TF (Term Frequency) measures
- What IDF (Inverse Document Frequency) measures
- How TF × IDF = a powerful relevance score
- Build a working search engine from scratch

---

## The Problem: Why Counting Words Fails

Imagine you have 3 documents and someone searches for **"pizza recipe"**.

A naive approach: count how many times each search word appears in each document.

But there's a flaw — the word **"the"** appears in almost every document. If you count it, documents with lots of "the" will rank higher even if they're completely irrelevant.

**TF-IDF solves this by:**
- ✅ Rewarding words that appear often **in this specific document** (TF)
- ✅ Penalizing words that appear in **every document** (IDF)
- ✅ Multiplying them together to get a final relevance score

In [ ]:
# Our mini document collection — 10 food/cooking articles
# Think of these as web pages returned by a crawler

documents = {
    "S1":  "The best homemade pizza recipe for beginners",
    "S2":  "How to make the best pizza dough from scratch recipe",
    "S3":  "Top 10 tips for baking pizza at home for beginners",
    "S4":  "Easy pasta recipes and Italian cooking techniques",
    "S5":  "A guide to building your own outdoor pizza oven in summer",
    "S6":  "The history of pizza from Naples to New York for foodies",
    "S7":  "Healthy meal prep ideas and cooking tips for temperature control",
    "S8":  "Best wood-fired pizza restaurants in the best cities",
    "S9":  "How to choose the right flour for homemade bread and pizza",
    "S10": "Italian cheese varieties and their uses in traditional cooking",
}

print(f"Total documents: {len(documents)}")
print()
for doc_id, text in documents.items():
    print(f"  {doc_id:4s}: {text}")

---

## STEP 1 — Demonstrate the Naive Counting Problem

Before building TF-IDF, let's see WHY simple counting fails.

We'll count how many times each word in the query `"best pizza recipe"` appears in each document — and rank by raw count.

In [ ]:
# Naive approach: just count query word occurrences

query_words = ["best", "pizza", "recipe"]

print("Naive word count scores for query: 'best pizza recipe'")
print(f"{'Doc':<5} {'best':>6} {'pizza':>6} {'recipe':>8} {'TOTAL':>7}")
print("-" * 38)

naive_scores = {}
for doc_id, text in documents.items():
    words = text.lower().split()            # split into lowercase words
    count_best   = words.count("best")
    count_pizza  = words.count("pizza")
    count_recipe = words.count("recipe")
    total = count_best + count_pizza + count_recipe
    naive_scores[doc_id] = total
    if total > 0:
        print(f"{doc_id:<5} {count_best:>6} {count_pizza:>6} {count_recipe:>8} {total:>7}")

print()
print("Ranked by naive count:")
for rank, (doc_id, score) in enumerate(
    sorted(naive_scores.items(), key=lambda x: -x[1])[:5], 1
):
    print(f"  {rank}. {doc_id} (count={score}) — {documents[doc_id]}")

print()
print("⚠️  Problem: S8 ranks #1 only because 'best' appears TWICE.")
print("   But S1 is clearly the most relevant: 'best homemade PIZZA RECIPE for beginners'")

---

## STEP 2 — Text Preprocessing (Getting Documents Ready)

Before calculating TF-IDF, we need to clean the text. Raw text is messy:
- "Recipe" and "recipe" should be the same word → **lowercase**
- "the", "a", "and" are noise → **remove stop words**
- "recipes" and "recipe" mean the same thing → **stemming**

Think of preprocessing as washing vegetables before cooking — you can't skip it.

In [ ]:
# --- Step 2a: Tokenise (split text into individual words) ---

def tokenise(text):
    # 👇 .lower() makes 'Pizza' == 'pizza', then .split() splits on spaces
    return text.lower().split()

example = documents["S1"]
tokens = tokenise(example)
print(f"Original : {example}")
print(f"Tokenised: {tokens}")

In [ ]:
# --- Step 2b: Remove stop words ---
# Stop words are high-frequency, low-meaning words.
# They appear in EVERY document, so they have zero discriminating power.

STOP_WORDS = {
    'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at',
    'to', 'for', 'of', 'is', 'it', 'from', 'how', 'your',
    'their', 'own', 'that', 'this'
}

def remove_stop_words(tokens):
    # 👇 Keep only tokens that are NOT in the stop words set
    return [t for t in tokens if t not in STOP_WORDS]

tokens   = tokenise(documents["S1"])
filtered = remove_stop_words(tokens)

print(f"Before stop word removal: {tokens}")
print(f"After  stop word removal: {filtered}")
removed = set(tokens) - set(filtered)
print(f"Removed: {removed}  ← these words tell us nothing useful")

In [ ]:
# --- Step 2c: Stemming ---
# "recipes", "recipe" → same root "recip"
# "cooking", "cooked", "cook" → same root "cook"
#
# We use a manual lookup table here (a real system uses NLTK's PorterStemmer)
# The goal: treat different forms of the same word as ONE word

STEM_RULES = {
    'recipes': 'recip', 'recipe': 'recip',
    'baking': 'bake',   'cooking': 'cook',
    'homemade': 'homemad', 'beginners': 'beginn',
    'building': 'build',   'cities': 'citi',
    'varieties': 'varieti', 'ideas': 'idea',
    'techniques': 'techniqu', 'traditional': 'tradit',
    'restaurants': 'restaur', 'healthy': 'healthi',
    'uses': 'use',     'tips': 'tip',
    'choose': 'choos', 'fired': 'fire',
}

def stem(token):
    # 👇 If we have a rule for this word, use the root; otherwise keep as-is
    return STEM_RULES.get(token, token)

# Show stemming in action
demo_words = ['recipes', 'recipe', 'baking', 'cooking', 'beginners', 'techniques']
print("Stemming examples:")
for w in demo_words:
    print(f"  {w:15s} → {stem(w)}")

print()
print("Why this matters: a search for 'recipe' will now match documents")
print("that say 'recipes' — because both map to 'recip'")

In [ ]:
# --- Step 2d: Full pipeline — combine all 3 steps ---

def process(text):
    tokens  = tokenise(text)             # Step 1: lowercase + split
    cleaned = remove_stop_words(tokens)  # Step 2: remove noise words
    stemmed = [stem(t) for t in cleaned] # Step 3: collapse word variants
    return stemmed

# Process ALL documents and store the result
processed = {}
for doc_id, text in documents.items():
    processed[doc_id] = process(text)

print("Processed documents (these are what TF-IDF works on):")
print()
for doc_id, terms in processed.items():
    print(f"  {doc_id:4s}: {terms}")

---

## STEP 3 — TF: Term Frequency

**Formula:**
```
TF(word, document) = (number of times word appears in document)
                     ─────────────────────────────────────────
                     (total number of words in document)
```

**Why divide by total words?**  
A 1000-word document will naturally have higher counts than a 10-word document. Dividing normalizes for document length — we care about *proportion*, not raw count.

**Range:** 0.0 (word not in doc) to 1.0 (document is one repeated word)

In [ ]:
# TF = count of word in this doc / total words in this doc

def tf(term, doc_terms):
    """
    term      : the word we're scoring (already stemmed/processed)
    doc_terms : list of processed tokens in the document
    """
    count      = doc_terms.count(term)  # 👇 how many times does 'term' appear?
    total      = len(doc_terms)         # 👇 total words in this document
    return count / total                # 👇 proportion of this doc that is 'term'

# --- Demonstration ---
print("TF scores for the word 'pizza' across all documents:")
print()
print(f"  {'Doc':<5} {'Tokens':<45} {'pizza count':>12} {'total words':>12} {'TF':>8}")
print("  " + "-" * 85)

for doc_id, terms in processed.items():
    score = tf("pizza", terms)
    count = terms.count("pizza")
    marker = " ←" if score > 0 else ""
    print(f"  {doc_id:<5} {str(terms):<45} {count:>12} {len(terms):>12} {score:>8.4f}{marker}")

print()
print("💡 S8 has the HIGHEST TF for 'pizza' even though it only appears once.")
print("   That's because S8 has fewer total words — so 'pizza' is a larger proportion.")

In [ ]:
# Let's also look at TF for 'best' — this will reveal TF's weakness

print("TF scores for the word 'best':")
print()
for doc_id, terms in processed.items():
    score = tf("best", terms)
    if score > 0:
        print(f"  {doc_id}: TF = {score:.4f}  (appears {terms.count('best')}x in {len(terms)} words) — {documents[doc_id]}")

print()
print("⚠️  TF alone still has a problem:")
print("   TF doesn't know whether 'best' appears in many documents or just a few.")
print("   A word that appears in ALL 10 docs tells us nothing about which doc to pick.")
print("   That's what IDF fixes.")

---

## STEP 4 — IDF: Inverse Document Frequency

**Formula:**
```
IDF(word) = log( total number of documents )
                 ─────────────────────────
                 number of docs containing word
```

**Intuition:**
- If a word appears in 1 out of 10 docs → `log(10/1) = 2.3` → HIGH score (rare = informative)
- If a word appears in 10 out of 10 docs → `log(10/10) = 0` → ZERO score (everywhere = useless)

**Why `log`?**  
Without log, a word in 1 doc would score 10, and a word in 2 docs would score 5. That's too aggressive. Log compresses the scale — the difference between 1 doc and 2 docs is much less dramatic than the difference between 1 and 1000.

In [ ]:
import math

def idf(term, all_docs):
    """
    term     : the word we're scoring
    all_docs : dict of {doc_id: [processed terms]}
    """
    N  = len(all_docs)                                       # 👇 total number of documents
    df = sum(1 for doc in all_docs.values() if term in doc)  # 👇 docs that contain this term
    if df == 0:
        return 0                                             # 👇 word never appears → score 0
    return math.log(N / df)                                  # 👇 log(N/df) = IDF

# --- See IDF for various words ---
words_to_check = ['pizza', 'best', 'recip', 'cook', 'beginn', 'foodi', 'naples', 'the']

print(f"IDF scores across {len(processed)} documents:")
print()
print(f"  {'Word':<15} {'Docs with it':>13} {'IDF score':>10}  Meaning")
print("  " + "-" * 65)

for word in words_to_check:
    df    = sum(1 for doc in processed.values() if word in doc)
    score = idf(word, processed)
    if df > 0:
        meaning = "very common → low signal" if score < 0.5 else (
                  "common"                   if score < 1.0 else (
                  "moderately rare"          if score < 2.0 else
                  "RARE → high signal"))
        print(f"  {word:<15} {df:>5}/{len(processed):<8} {score:>10.4f}  {meaning}")

In [ ]:
# Visual proof: why log() matters
# Without log, the score would be too extreme

N = 10  # our 10 documents

print("Effect of log() on IDF scores:")
print()
print(f"  {'Docs with word':>15} {'Raw N/df':>10} {'With log()':>12}  Interpretation")
print("  " + "-" * 65)

for df in [10, 8, 5, 3, 2, 1]:
    raw  = N / df
    with_log = math.log(N / df)
    interp = "common" if df >= 7 else ("moderate" if df >= 3 else "rare")
    print(f"  {df:>5}/{N:<10} {raw:>10.2f} {with_log:>12.4f}  {interp}")

print()
print("💡 Without log: word in 1 doc scores 10.0, word in 2 docs scores 5.0 (2x difference)")
print("   With log:    word in 1 doc scores 2.30, word in 2 docs scores 1.61 (much gentler)")
print("   Log prevents rare words from completely dominating the score.")

---

## STEP 5 — TF-IDF = TF × IDF

Now we multiply them together:

```
TF-IDF(word, document) = TF(word, document)  ×  IDF(word)
```

**What each combination means:**

| TF | IDF | TF-IDF | Meaning |
|---|---|---|---|
| High | High | **High** | Word appears often in THIS doc AND is rare globally → very relevant |
| High | Low | **Low** | Common word like "the" — appears everywhere, meaningless |
| Low | High | **Medium** | Rare word but barely used in this doc → weak signal |
| Low | Low | **Zero** | Common word barely used — total noise |

In [ ]:
def tfidf(term, doc_terms, all_docs):
    """
    Final TF-IDF score for a term in a document.
    High score = this term is important TO THIS DOCUMENT specifically.
    """
    return tf(term, doc_terms) * idf(term, all_docs)

# --- Deep dive: score 'pizza' vs 'best' across all documents ---

print("TF-IDF comparison: 'pizza' vs 'best'")
print()
print(f"  {'IDF pizza':>10} = {idf('pizza', processed):.4f}  (pizza in {sum(1 for d in processed.values() if 'pizza' in d)}/10 docs)")
print(f"  {'IDF best' :>10} = {idf('best',  processed):.4f}  (best  in {sum(1 for d in processed.values() if 'best'  in d)}/10 docs)")
print()
print(f"  {'Doc':<5} {'TF(pizza)':>10} {'TF-IDF(pizza)':>14} {'TF(best)':>10} {'TF-IDF(best)':>13}")
print("  " + "-" * 55)

for doc_id, terms in processed.items():
    tf_pizza   = tf("pizza", terms)
    tf_best    = tf("best",  terms)
    tfidf_pizza = tfidf("pizza", terms, processed)
    tfidf_best  = tfidf("best",  terms, processed)
    if tf_pizza > 0 or tf_best > 0:
        print(f"  {doc_id:<5} {tf_pizza:>10.4f} {tfidf_pizza:>14.4f} {tf_best:>10.4f} {tfidf_best:>13.4f}")

print()
print("💡 Notice: even though 'pizza' appears in more docs than 'best',")
print("   the TF-IDF scores properly weight both based on their rarity.")

In [ ]:
# --- Score a full query: sum TF-IDF over all query terms ---
# For query "best pizza recipe":
#   score(doc) = TF-IDF("best", doc) + TF-IDF("pizza", doc) + TF-IDF("recip", doc)

query_raw   = "best pizza recipe"
query_terms = process(query_raw)   # run same preprocessing on the query!
print(f"Query: '{query_raw}'")
print(f"After preprocessing: {query_terms}")
print()

print(f"  {'Doc':<5} ", end="")
for t in query_terms:
    print(f" {t:>10}", end="")
print(f" {'TOTAL':>10}")
print("  " + "-" * (5 + 10 * len(query_terms) + 12))

all_scores = {}
for doc_id, terms in processed.items():
    per_term = {t: tfidf(t, terms, processed) for t in query_terms}
    total    = sum(per_term.values())
    all_scores[doc_id] = total
    if total > 0:
        print(f"  {doc_id:<5} ", end="")
        for t in query_terms:
            print(f" {per_term[t]:>10.4f}", end="")
        print(f" {total:>10.4f}")

print()
print("🏆 Ranked Results:")
for rank, (doc_id, score) in enumerate(
    sorted(all_scores.items(), key=lambda x: -x[1])[:5], 1
):
    print(f"  {rank}. {doc_id} (score: {score:.4f}) — {documents[doc_id]}")

---

## STEP 6 — Build a Complete Search Function

Let's wrap everything into a clean `search()` function that:
1. Preprocesses the query (same pipeline as documents)
2. Uses an inverted index to find candidate documents fast
3. Scores candidates with TF-IDF
4. Returns ranked results

In [ ]:
from collections import defaultdict

# --- Build an inverted index for fast lookup ---
# Instead of scanning all documents for every query,
# we pre-build a map: word → which documents contain it

inverted_index = defaultdict(set)           # maps term → set of doc_ids
for doc_id, terms in processed.items():
    for term in terms:
        inverted_index[term].add(doc_id)    # 👇 record that this doc contains this term

# Show the index for a few terms
print("Inverted index (term → documents that contain it):")
print()
for term in ['pizza', 'recip', 'cook', 'best', 'beginn', 'naples']:
    docs = sorted(inverted_index[term])
    print(f"  {term:<12} → {docs}")

print()
print("💡 For a query, we union the sets for each query word.")
print("   query 'pizza recipe' → pizza_docs ∪ recipe_docs = candidate set")

In [ ]:
def search(query_text, top_k=5, verbose=True):
    """
    Full TF-IDF search:
    1. Preprocess query  (same pipeline as documents)
    2. Lookup candidates via inverted index  (fast!)
    3. Score each candidate with TF-IDF
    4. Return top_k ranked results
    """
    # Step 1: preprocess the query the same way we processed documents
    query_terms = process(query_text)

    # Step 2: find candidate documents using the inverted index
    # Union of all doc sets for each query term
    candidates = set()
    for term in query_terms:
        candidates |= inverted_index.get(term, set())  # 👇 |= is set union

    # Step 3: score each candidate
    scores = {}
    for doc_id in candidates:
        # Sum TF-IDF score for EACH query term in this document
        scores[doc_id] = sum(
            tfidf(t, processed[doc_id], processed) for t in query_terms
        )

    # Step 4: sort by score descending, take top k
    ranked = sorted(scores.items(), key=lambda x: -x[1])[:top_k]

    if verbose:
        print(f"Query: '{query_text}'")
        print(f"Preprocessed: {query_terms}")
        print(f"Candidates: {sorted(candidates)} ({len(candidates)} docs to score)")
        print()
        print(f"Top {top_k} Results:")
        for rank, (doc_id, score) in enumerate(ranked, 1):
            print(f"  {rank}. [{score:.4f}] {documents[doc_id]}")
        print()

    return ranked

In [ ]:
# --- Try different queries ---

print("=" * 60)
search("best pizza recipe")

print("=" * 60)
search("Italian traditional cooking")

print("=" * 60)
search("homemade bread flour")

print("=" * 60)
search("healthy meal ideas")

---

## STEP 7 — Where TF-IDF Fails

TF-IDF is powerful but has known limitations. Understanding these failures is exactly how you know when to move to something better.

In [ ]:
# FAILURE 1: Synonyms are invisible to TF-IDF
# TF-IDF treats words as exact strings — 'car' and 'automobile' are completely different.

print("FAILURE 1: Synonyms")
print("Query: 'automobile' — but all our documents say 'car' or 'vehicle'")

# Add a test doc
test_docs = {
    "D1": "car racing tips and tricks",
    "D2": "vehicle maintenance guide",
    "D3": "automobile history and evolution",
}
processed_test = {k: process(v) for k, v in test_docs.items()}
inv_test = defaultdict(set)
for doc_id, terms in processed_test.items():
    for t in terms: inv_test[t].add(doc_id)

query_terms = process("car")
print(f"  Query 'car' preprocessed: {query_terms}")
print(f"  Docs containing 'car': {sorted(inv_test.get('car', set()))}")
print(f"  Docs containing 'automobil': {sorted(inv_test.get('automobil', set()))}")
print()
print("  'car' matches D1 but not D3 ('automobile')")
print("  Even though D3 is clearly about the same topic!")
print()
print("→ FIX: Word embeddings (Word2Vec, BERT) understand that car ≈ automobile")

In [ ]:
# FAILURE 2: Word order doesn't matter
# 'dog bites man' and 'man bites dog' have the SAME TF-IDF scores

print("FAILURE 2: Word order is ignored")
print()

sent1 = "the dog bites the man"
sent2 = "the man bites the dog"

tokens1 = process(sent1)
tokens2 = process(sent2)

print(f"Sentence 1 tokens: {tokens1}")
print(f"Sentence 2 tokens: {tokens2}")
print()

# The token bags are identical (same words, different order)
print(f"Same tokens? {sorted(tokens1) == sorted(tokens2)}")
print()
print("TF-IDF gives IDENTICAL scores to both sentences for any query.")
print("It has no concept of meaning, syntax, or order.")
print()
print("→ FIX: Transformers (BERT, GPT) read the full sequence in order")

In [ ]:
# FAILURE 3: New words have zero IDF
# If a word never appeared in training documents, IDF = 0
# Called the 'Out of Vocabulary' (OOV) problem

print("FAILURE 3: Out-of-vocabulary (OOV) words")
print()

new_word = "sourdough"
df = sum(1 for doc in processed.values() if new_word in doc)
score = idf(new_word, processed)

print(f"  Word: '{new_word}'")
print(f"  Appears in {df} documents")
print(f"  IDF = {score}")
print(f"  TF-IDF for any doc = 0 × 0 = 0")
print()
print("  If someone searches for 'sourdough pizza', we return nothing!")
print()
print("→ FIX: BM25 handles this with smoothing; embeddings generalize to unseen words")

---

## STEP 8 — Summary: What TF-IDF Does (and Doesn't Do)

| | TF-IDF | Word Embeddings (next) |
|---|---|---|
| Handles exact word matches | ✅ Yes | ✅ Yes |
| Handles synonyms | ❌ No | ✅ Yes |
| Understands word order | ❌ No | ✅ Yes |
| Handles new/rare words | ❌ No | ✅ Yes |
| Fast to compute | ✅ Very fast | ⚠️ Slower |
| Easy to explain | ✅ Fully transparent | ❌ Black box |
| No GPU needed | ✅ Yes | ⚠️ Preferred |

**When to use TF-IDF:**
- Keyword search on large document collections
- When exact term matching matters (legal, medical, code search)
- As a fast first-pass filter before a slower re-ranker
- When you need full explainability (why did this doc rank here?)

**What's next:**  
Word2Vec and sentence embeddings turn words into vectors that *understand meaning* — so "pizza" and "Italian dish" become close in vector space. That's what M00 theory covers next.

In [ ]:
# Final recap: run the full pipeline end-to-end in one cell

print("=" * 55)
print(" FULL TF-IDF PIPELINE RECAP")
print("=" * 55)
print()

pipeline_steps = [
    ("1. Tokenise",         "Split text into lowercase words"),
    ("2. Stop words",       "Remove 'the', 'a', 'and', ..."),
    ("3. Stem",             "'recipes' → 'recip', 'cooking' → 'cook'"),
    ("4. Build index",      "Map each term → which docs contain it"),
    ("5. TF",               "Word count in this doc / total words in this doc"),
    ("6. IDF",              "log(total docs / docs with this word)"),
    ("7. TF-IDF",           "TF × IDF = relevance score"),
    ("8. Rank",             "Sum TF-IDF over query terms, sort descending"),
]
for step, desc in pipeline_steps:
    print(f"  {step:<20} {desc}")

print()
print("Try your own query:")
user_query = "homemade pizza beginners"   # 👈 change this to anything!
search(user_query)